In [1]:
import os
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from scipy.stats import pearsonr

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [2]:
BASE_DIR = "/content/drive/MyDrive/PPG_Project"

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "processed"
)

SPLIT_DIR = os.path.join(
    BASE_DIR,
    "splits"
)

CHECKPOINT_DIR = os.path.join(
    BASE_DIR,
    "checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

In [3]:
import sys
import os

# everything in this cell

In [4]:
!git clone https://github.com/yuqinie98/PatchTST.git

fatal: destination path 'PatchTST' already exists and is not an empty directory.


In [5]:
import sys

PATCHTST_PATH = "/content/PatchTST/PatchTST_supervised"

if PATCHTST_PATH not in sys.path:
    sys.path.insert(0, PATCHTST_PATH)

print(sys.path[0])

/content/PatchTST/PatchTST_supervised


In [6]:
from layers.PatchTST_backbone import TSTiEncoder

print("SUCCESS")

SUCCESS


In [7]:
X_ppg = np.load(
    os.path.join(
        PROCESSED_DIR,
        "X_ppg.npy"
    )
)

y = np.load(
    os.path.join(
        PROCESSED_DIR,
        "y.npy"
    )
)

train_idx = np.load(
    os.path.join(
        SPLIT_DIR,
        "train_idx.npy"
    )
)

val_idx = np.load(
    os.path.join(
        SPLIT_DIR,
        "val_idx.npy"
    )
)

test_idx = np.load(
    os.path.join(
        SPLIT_DIR,
        "test_idx.npy"
    )
)

print(X_ppg.shape)
print(y.shape)

(64697, 512)
(64697,)


In [8]:
class PPGDataset(Dataset):

    def __init__(
        self,
        X,
        y,
        indices,
        train=False
    ):

        self.X = X[indices]
        self.y = y[indices]

        self.train = train

    def __len__(self):

        return len(self.y)

    def __getitem__(self, idx):

        signal = self.X[idx].copy()

        target = self.y[idx]

        if self.train:

            # Gaussian Noise

            if np.random.rand() < 0.5:

                signal += np.random.normal(
                    0,
                    0.01,
                    signal.shape
                )

            # Amplitude Scaling

            if np.random.rand() < 0.5:

                scale = np.random.uniform(
                    0.9,
                    1.1
                )

                signal *= scale

        signal = torch.FloatTensor(signal)

        # PatchTST expects:
        #
        # (seq_len, channels)

        signal = signal.unsqueeze(-1)

        target = torch.FloatTensor(
            [target]
        )

        return signal, target

In [9]:
BATCH_SIZE = 64

train_dataset = PPGDataset(
    X_ppg,
    y,
    train_idx,
    train=True
)

val_dataset = PPGDataset(
    X_ppg,
    y,
    val_idx,
    train=False
)

test_dataset = PPGDataset(
    X_ppg,
    y,
    test_idx,
    train=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(len(train_loader))

808


In [10]:
x, y_batch = next(
    iter(train_loader)
)

print(x.shape)
print(y_batch.shape)

torch.Size([64, 512, 1])
torch.Size([64, 1])


In [11]:
# =====================================================
# Model Hyperparameters
# =====================================================

SEQ_LEN = 512

PATCH_LEN = 16
STRIDE = 8

PATCH_NUM = int(
    (SEQ_LEN - PATCH_LEN)/STRIDE + 1
)

D_MODEL = 128

N_HEADS = 8

N_LAYERS = 4

D_FF = 256

DROPOUT = 0.1

print("Patch Num =", PATCH_NUM)

Patch Num = 63


In [12]:
# =====================================================
# Heart Rate PatchTST
# =====================================================

import torch
import torch.nn as nn


class HeartRatePatchTST(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = TSTiEncoder(
            c_in=1,
            patch_num=PATCH_NUM,
            patch_len=PATCH_LEN,

            n_layers=N_LAYERS,
            d_model=D_MODEL,
            n_heads=N_HEADS,
            d_ff=D_FF,

            dropout=DROPOUT
        )

        feature_dim = D_MODEL * PATCH_NUM

        self.regressor = nn.Sequential(

            nn.Flatten(),

            nn.Linear(feature_dim, 256),

            nn.GELU(),

            nn.Dropout(0.2),

            nn.Linear(256, 64),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        # x:
        # [B,512,1]

        x = x.permute(
            0,
            2,
            1
        )

        x = x.unfold(
            dimension=-1,
            size=PATCH_LEN,
            step=STRIDE
        )

        # [B,1,63,16]

        x = x.permute(
            0,
            1,
            3,
            2
        )

        # [B,1,16,63]

        features = self.encoder(x)

        prediction = self.regressor(
            features
        )

        return prediction

In [13]:
# =====================================================
# Create Model
# =====================================================

model = HeartRatePatchTST()

model = model.to(device)

print(model.__class__.__name__)

HeartRatePatchTST


In [14]:
# =====================================================
# Test Model
# =====================================================

sample_x, sample_y = next(
    iter(train_loader)
)

sample_x = sample_x.to(device)

with torch.no_grad():

    pred = model(sample_x)

print("Prediction Shape:", pred.shape)

Prediction Shape: torch.Size([64, 1])


In [15]:
# =====================================================
# Loss Function
# =====================================================

criterion = nn.L1Loss()

print(criterion)

L1Loss()


In [16]:
# =====================================================
# AdamW Optimizer
# =====================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [17]:
# =====================================================
# Reduce LR on Plateau
# =====================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

In [18]:
# =====================================================
# Metrics
# =====================================================

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from scipy.stats import pearsonr

import numpy as np


def compute_metrics(
    y_true,
    y_pred
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    corr, _ = pearsonr(
        y_true,
        y_pred
    )

    return mae, rmse, corr

In [19]:
# =====================================================
# Early Stopping
# =====================================================

class EarlyStopping:

    def __init__(
        self,
        patience=10
    ):

        self.patience = patience

        self.counter = 0

        self.best_loss = np.inf

        self.stop = False

    def step(
        self,
        loss
    ):

        if loss < self.best_loss:

            self.best_loss = loss

            self.counter = 0

            return True

        else:

            self.counter += 1

            if self.counter >= self.patience:

                self.stop = True

            return False

In [20]:
# =====================================================
# Validation
# =====================================================

def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    losses = []

    preds = []

    trues = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)

            y = y.to(device)

            pred = model(x)

            loss = criterion(
                pred,
                y
            )

            losses.append(
                loss.item()
            )

            preds.extend(
                pred.cpu().numpy().flatten()
            )

            trues.extend(
                y.cpu().numpy().flatten()
            )

    mae, rmse, corr = compute_metrics(
        trues,
        preds
    )

    return (
        np.mean(losses),
        mae,
        rmse,
        corr
    )

In [21]:
# =====================================================
# Save Folder
# =====================================================

CHECKPOINT_DIR = (
    "/content/drive/MyDrive/PPG_Project/checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "patchtst_hr_best.pth"
)

print(BEST_MODEL_PATH)

/content/drive/MyDrive/PPG_Project/checkpoints/patchtst_hr_best.pth


In [22]:
# =====================================================
# Training Loop
# =====================================================

EPOCHS = 50

early_stopping = EarlyStopping(
    patience=10
)

for epoch in range(EPOCHS):

    model.train()

    train_losses = []

    for x, y in train_loader:

        x = x.to(device)

        y = y.to(device)

        optimizer.zero_grad()

        pred = model(x)

        loss = criterion(
            pred,
            y
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        train_losses.append(
            loss.item()
        )

    train_loss = np.mean(
        train_losses
    )

    val_loss, val_mae, val_rmse, val_corr = evaluate(
        model,
        val_loader,
        criterion
    )

    scheduler.step(
        val_loss
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train {train_loss:.4f} | "
        f"Val MAE {val_mae:.4f} | "
        f"Val RMSE {val_rmse:.4f} | "
        f"Val Corr {val_corr:.4f}"
    )

    if early_stopping.step(
        val_loss
    ):

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print(
            "Best model saved."
        )

    if early_stopping.stop:

        print(
            "Early stopping."
        )

        break

Epoch 01 | Train 28.6009 | Val MAE 12.1848 | Val RMSE 16.0911 | Val Corr 0.7484
Best model saved.
Epoch 02 | Train 14.5790 | Val MAE 6.7383 | Val RMSE 10.4491 | Val Corr 0.8834
Best model saved.
Epoch 03 | Train 13.2405 | Val MAE 8.0094 | Val RMSE 10.9600 | Val Corr 0.8879
Epoch 04 | Train 12.7636 | Val MAE 6.4760 | Val RMSE 9.2908 | Val Corr 0.9077
Best model saved.
Epoch 05 | Train 12.2928 | Val MAE 7.0329 | Val RMSE 9.9859 | Val Corr 0.8979
Epoch 06 | Train 12.1159 | Val MAE 6.5344 | Val RMSE 9.3724 | Val Corr 0.9053
Epoch 07 | Train 11.9781 | Val MAE 6.5300 | Val RMSE 9.4732 | Val Corr 0.9036
Epoch 08 | Train 11.7878 | Val MAE 6.5729 | Val RMSE 9.5536 | Val Corr 0.9026
Epoch 09 | Train 11.5002 | Val MAE 5.9737 | Val RMSE 8.7972 | Val Corr 0.9191
Best model saved.
Epoch 10 | Train 11.3485 | Val MAE 5.9700 | Val RMSE 8.7850 | Val Corr 0.9174
Epoch 11 | Train 11.3115 | Val MAE 6.5890 | Val RMSE 9.6600 | Val Corr 0.9071
Epoch 12 | Train 11.1894 | Val MAE 6.5671 | Val RMSE 9.3363 | Val 

In [23]:
# =====================================================
# Load Best Model
# =====================================================

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH
    )
)

print("Best model loaded.")

Best model loaded.


In [26]:
model.load_state_dict(
    torch.load(BEST_MODEL_PATH)
)

model.eval()

HeartRatePatchTST(
  (encoder): TSTiEncoder(
    (W_P): Linear(in_features=16, out_features=128, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (encoder): TSTEncoder(
      (layers): ModuleList(
        (0-3): 4 x TSTEncoderLayer(
          (self_attn): _MultiheadAttention(
            (W_Q): Linear(in_features=128, out_features=128, bias=True)
            (W_K): Linear(in_features=128, out_features=128, bias=True)
            (W_V): Linear(in_features=128, out_features=128, bias=True)
            (sdp_attn): _ScaledDotProductAttention(
              (attn_dropout): Dropout(p=0.0, inplace=False)
            )
            (to_out): Sequential(
              (0): Linear(in_features=128, out_features=128, bias=True)
              (1): Dropout(p=0.1, inplace=False)
            )
          )
          (dropout_attn): Dropout(p=0.1, inplace=False)
          (norm_attn): Sequential(
            (0): Transpose()
            (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=Tru

In [27]:
# ==========================================
# Predict one sample
# ==========================================

model.eval()

x, y_true = test_dataset[0]

with torch.no_grad():

    pred = model(
        x.unsqueeze(0).to(device)
    )

print("Actual HR:", y_true.item())
print("Predicted HR:", pred.item())
print("Error:", abs(pred.item() - y_true.item()))

Actual HR: 72.67472839355469
Predicted HR: 69.65900421142578
Error: 3.0157241821289062


In [24]:
# =====================================================
# Test Set Evaluation
# =====================================================

test_loss, test_mae, test_rmse, test_corr = evaluate(
    model,
    test_loader,
    criterion
)

print("\nFINAL TEST RESULTS")

print(
    f"MAE   : {test_mae:.4f}"
)

print(
    f"RMSE  : {test_rmse:.4f}"
)

print(
    f"CORR  : {test_corr:.4f}"
)


FINAL TEST RESULTS
MAE   : 7.1525
RMSE  : 10.9435
CORR  : 0.8431


In [25]:
import numpy as np
import os

X_acc_xyz = np.load(
    os.path.join(
        PROCESSED_DIR,
        "X_acc_xyz.npy"
    )
)

print(X_acc_xyz.shape)

(64697, 256, 3)
